# SVG Scaling Laws — Full Training Pipeline

**Runtime**: A100 GPU (Colab Pro)

This notebook runs the entire experiment pipeline:

| # | Section | Est. Time |
|---|---------|----------|
| 1 | Setup (Drive + git pull) | 1 min |
| 2 | Data preprocessing + tokenization | 15 min |
| 3 | SP LR Sweep (Tiny × 5 LRs) | 15 min |
| 4 | SP Scaling Study (5 models) | 2.5 h |
| 5 | µP LR Sweep (Tiny × 7 LRs) | 20 min |
| 6 | µP Scaling Study (5 models) | 2.5 h |
| 7 | Best model extended training | 1.5 h |
| 8 | Generation + Evaluation | 15 min |

**Total: ~7 hours** (set Runtime → A100, then Run All)

Results are saved on Drive so they survive session disconnects.
Each section checks for existing results and skips if already complete.

---
## 1. Setup

In [11]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/svg-scaling-project
!git pull
!pip install -r requirements.txt -q

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/svg-scaling-project
Already up to date.


In [12]:
import torch, subprocess, time, json, os, math

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

def run_train(args, label):
    """Run train.py and return summary dict."""
    t0 = time.time()
    result = subprocess.run(['python', 'src/train.py'] + args)
    elapsed = time.time() - t0
    status = 'OK' if result.returncode == 0 else f'FAIL({result.returncode})'
    print(f'  {label}: {status}, {elapsed/60:.1f} min')
    return result.returncode

def load_summary(output_dir):
    """Load summary.json from a run directory."""
    path = os.path.join(output_dir, 'summary.json')
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)

def is_done(output_dir):
    """Check if a run already completed."""
    return os.path.exists(os.path.join(output_dir, 'summary.json'))

CUDA: True
GPU: NVIDIA A100-SXM4-40GB
Memory: 42.4 GB


---
## 2. Data Preprocessing + Tokenization

Downloads `starvector/svg-icons-simple` from HuggingFace, cleans SVGs, trains BPE tokenizer.

If tokens < 100M (due to coordinate normalization shortening SVGs), automatically adds `svg-emoji-simple` as supplementary data.

In [13]:
import shutil

def tokenize_and_check():
    """Tokenize and return train token count."""
    if os.path.exists('data/tokenized'):
        shutil.rmtree('data/tokenized')
    subprocess.run([
        'python', 'src/tokenize_data.py',
        '--input-dir', 'data/processed',
        '--output-dir', 'data/tokenized',
        '--vocab-size', '4096',
        '--max-token-len', '0',
    ])
    with open('data/tokenized/tokenize_stats.json') as f:
        return json.load(f)['train']['total_tokens']

# Step 1: Preprocess primary dataset
if not os.path.exists('data/processed/train.jsonl'):
    subprocess.run([
        'python', 'src/preprocess.py',
        '--download', 'starvector/svg-icons-simple',
        '--output-dir', 'data/processed',
        '--min-len', '50',
    ])

# Step 2: Tokenize
n_tokens = tokenize_and_check()
print(f"Train tokens: {n_tokens:,}")

# Step 3: If < 100M, add supplementary dataset
if n_tokens < 100_000_000:
    print(f"\n{n_tokens:,} < 100M. Adding svg-emoji-simple...")
    subprocess.run([
        'python', 'src/preprocess.py',
        '--download', 'starvector/svg-emoji-simple',
        '--output-dir', 'data/processed_emoji',
        '--min-len', '50',
    ])
    # Append to main dataset
    for split in ['train', 'val', 'test']:
        src = f'data/processed_emoji/{split}.jsonl'
        dst = f'data/processed/{split}.jsonl'
        if os.path.exists(src):
            with open(src) as f_in, open(dst, 'a') as f_out:
                f_out.write(f_in.read())
            print(f"  Appended {split}.jsonl")

    # Re-tokenize with combined data
    n_tokens = tokenize_and_check()
    print(f"Train tokens after supplement: {n_tokens:,}")

assert n_tokens >= 100_000_000, f'Need >= 100M train tokens, got {n_tokens:,}'
print(f"\n✓ {n_tokens:,} train tokens (>= 100M)")

Train tokens: 98,149,972

98,149,972 < 100M. Adding svg-emoji-simple...
  Appended train.jsonl
  Appended val.jsonl
  Appended test.jsonl
Train tokens after supplement: 107,140,922

✓ 107,140,922 train tokens (>= 100M)


---
## 3. SP LR Sweep (Tiny × 5 LRs)

In [14]:
sp_lrs = ['3e-5', '1e-4', '3e-4', '1e-3', '3e-3']
sp_sweep_results = []

print('=== SP LR Sweep ===')
for lr in sp_lrs:
    out = f'results/runs/lr_sweep/tiny_lr_{lr}'
    if is_done(out):
        print(f'  LR={lr}: already done, skipping')
    else:
        run_train(['--config', 'configs/tiny.yaml',
                   '--learning-rate', lr,
                   '--output-dir', out,
                   '--device', 'cuda'], f'SP Tiny LR={lr}')
    s = load_summary(out)
    if s:
        sp_sweep_results.append({'lr': lr, **s})

# Find best
print(f'\n{"LR":>8s}  {"Final Val Loss":>14s}  {"Final PPL":>10s}')
print('-' * 36)
for r in sorted(sp_sweep_results, key=lambda x: x['final_val_loss']):
    print(f'{r["lr"]:>8s}  {r["final_val_loss"]:>14.4f}  {r["final_val_ppl"]:>10.2f}')

sp_best = min(sp_sweep_results, key=lambda x: x['final_val_loss'])
sp_lr = sp_best['lr']
print(f'\n→ SP optimal LR: {sp_lr} (final_val_loss={sp_best["final_val_loss"]:.4f})')

=== SP LR Sweep ===
  SP Tiny LR=3e-5: OK, 2.9 min
  SP Tiny LR=1e-4: OK, 2.7 min
  SP Tiny LR=3e-4: OK, 2.7 min
  SP Tiny LR=1e-3: OK, 2.7 min
  SP Tiny LR=3e-3: OK, 2.7 min

      LR  Final Val Loss   Final PPL
------------------------------------
    3e-3          0.4948        1.64
    1e-3          0.5538        1.74
    3e-4          0.6664        1.95
    1e-4          0.9329        2.54
    3e-5          1.3600        3.90

→ SP optimal LR: 3e-3 (final_val_loss=0.4948)


---
## 4. SP Scaling Study (5 Models)

In [15]:
models = ['tiny', 'small', 'medium', 'large', 'xl']
sp_results = []

print(f'=== SP Scaling Study (LR={sp_lr}) ===')
for name in models:
    out = f'results/runs/sp/{name}'
    if is_done(out):
        print(f'  {name}: already done, skipping')
    else:
        run_train(['--config', f'configs/{name}.yaml',
                   '--learning-rate', sp_lr,
                   '--output-dir', out,
                   '--device', 'cuda'], f'SP {name}')
    s = load_summary(out)
    if s:
        sp_results.append(s)

print(f'\n{"Model":>8s}  {"Params":>8s}  {"Final Val Loss":>14s}  {"Final PPL":>10s}')
print('-' * 46)
for r in sp_results:
    print(f'{r["config_name"]:>8s}  {r["n_params"]/1e6:>7.1f}M  '
          f'{r["final_val_loss"]:>14.4f}  {r["final_val_ppl"]:>10.2f}')

=== SP Scaling Study (LR=3e-3) ===
  SP tiny: OK, 2.7 min
  SP small: OK, 5.6 min
  SP medium: OK, 12.5 min
  SP large: OK, 30.3 min
  SP xl: OK, 68.9 min

   Model    Params  Final Val Loss   Final PPL
----------------------------------------------
    tiny      1.3M          0.4796        1.62
   small      3.4M          0.4071        1.50
  medium     12.2M          0.4013        1.49
   large     33.6M          1.3050        3.69
      xl     88.1M          1.6668        5.30


---
## 5. µP LR Sweep (Tiny × 7 LRs)

Higher LRs (1e-2, 3e-2) included — µP should stabilize them.

In [16]:
mup_lrs = ['3e-5', '1e-4', '3e-4', '1e-3', '3e-3', '1e-2', '3e-2']
mup_sweep_results = []

print('=== µP LR Sweep ===')
for lr in mup_lrs:
    out = f'results/runs/mup_lr_sweep/tiny_lr_{lr}'
    if is_done(out):
        print(f'  LR={lr}: already done, skipping')
    else:
        run_train(['--config', 'configs/tiny.yaml',
                   '--learning-rate', lr,
                   '--output-dir', out,
                   '--device', 'cuda',
                   '--mup'], f'µP Tiny LR={lr}')
    s = load_summary(out)
    if s:
        mup_sweep_results.append({'lr': lr, **s})

print(f'\n{"LR":>8s}  {"Final Val Loss":>14s}  {"Final PPL":>10s}')
print('-' * 36)
for r in sorted(mup_sweep_results, key=lambda x: x['final_val_loss']):
    print(f'{r["lr"]:>8s}  {r["final_val_loss"]:>14.4f}  {r["final_val_ppl"]:>10.2f}')

mup_best = min(mup_sweep_results, key=lambda x: x['final_val_loss'])
mup_lr = mup_best['lr']
print(f'\n→ µP optimal LR: {mup_lr} (final_val_loss={mup_best["final_val_loss"]:.4f})')

=== µP LR Sweep ===
  µP Tiny LR=3e-5: OK, 2.7 min
  µP Tiny LR=1e-4: OK, 2.8 min
  µP Tiny LR=3e-4: OK, 2.8 min
  µP Tiny LR=1e-3: OK, 2.8 min
  µP Tiny LR=3e-3: OK, 2.7 min
  µP Tiny LR=1e-2: OK, 2.8 min
  µP Tiny LR=3e-2: OK, 2.7 min

      LR  Final Val Loss   Final PPL
------------------------------------
    1e-2          0.4517        1.57
    3e-2          0.5512        1.74
    3e-3          0.5551        1.74
    1e-3          0.6220        1.86
    3e-4          0.7217        2.06
    1e-4          1.0469        2.85
    3e-5          1.3049        3.69

→ µP optimal LR: 1e-2 (final_val_loss=0.4517)


---
## 6. µP Scaling Study (5 Models)

In [17]:
mup_results = []

print(f'=== µP Scaling Study (LR={mup_lr}) ===')
for name in models:
    out = f'results/runs/mup/{name}'
    if is_done(out):
        print(f'  {name}: already done, skipping')
    else:
        run_train(['--config', f'configs/{name}.yaml',
                   '--learning-rate', mup_lr,
                   '--output-dir', out,
                   '--device', 'cuda',
                   '--mup'], f'µP {name}')
    s = load_summary(out)
    if s:
        mup_results.append(s)

print(f'\n{"Model":>8s}  {"Params":>8s}  {"Final Val Loss":>14s}  {"Final PPL":>10s}')
print('-' * 46)
for r in mup_results:
    print(f'{r["config_name"]:>8s}  {r["n_params"]/1e6:>7.1f}M  '
          f'{r["final_val_loss"]:>14.4f}  {r["final_val_ppl"]:>10.2f}')

=== µP Scaling Study (LR=1e-2) ===
  µP tiny: OK, 2.8 min
  µP small: OK, 5.7 min
  µP medium: OK, 12.6 min
  µP large: OK, 31.6 min
  µP xl: OK, 70.7 min

   Model    Params  Final Val Loss   Final PPL
----------------------------------------------
    tiny      1.8M          0.4712        1.60
   small      4.2M          0.4198        1.52
  medium     13.8M          0.4324        1.54
   large     35.7M          0.4161        1.52
      xl     91.2M          0.3807        1.46


---
## 7. Best Model Extended Training (Part 4)

Resume µP XL from final checkpoint and train for additional epochs.

In [19]:
extended_dir = 'results/runs/mup_xl_extended'

if is_done(extended_dir):
    print('Extended training already done, skipping.')
else:
    # Resume from µP XL checkpoint, train for 2x the original steps
    xl_summary = load_summary('results/runs/mup/xl')
    original_steps = xl_summary['max_steps']
    extended_steps = original_steps * 2
    print(f'Original steps: {original_steps}, extending to: {extended_steps}')

    run_train(['--config', 'configs/xl.yaml',
               '--learning-rate', mup_lr,
               '--output-dir', extended_dir,
               '--device', 'cuda',
               '--mup',
               '--resume', 'results/runs/mup/xl/final_checkpoint.pt',
               '--max-steps', str(extended_steps)],
              f'µP XL extended ({extended_steps} steps)')

ext_summary = load_summary(extended_dir)
if ext_summary:
    print(f'\nExtended XL: final_val_loss={ext_summary["final_val_loss"]:.4f}, '
          f'ppl={ext_summary["final_val_ppl"]:.2f}')

Original steps: 6533, extending to: 13066
  µP XL extended (13066 steps): OK, 72.1 min

Extended XL: final_val_loss=0.3562, ppl=1.43


---
## 8. Generation + Evaluation (v2)

Uses `max_new_tokens=4096` (p99 coverage) and v2 directory layout.
- Unconditional top-p (0.95): 3 temps × 10 = 30 (matches v1 sampling for comparison)
- Unconditional top-k (40): 3 temps × 10 = 30
- Prefix-conditioned (top-p 0.95): 3 prefixes × 5 = 15

In [ ]:
import shutil as _shutil

# Checkpoint resolution — try extended first, then known µP XL locations
_ckpt_candidates = [
    f'{extended_dir}/best_model.pt',
    'results/runs/mup/xl/best_model.pt',
    'results/runs/mup_scaling_study/xl/best_model.pt',
]
ckpt = next((p for p in _ckpt_candidates if os.path.exists(p)), None)
assert ckpt, f'No checkpoint found. Tried:\n  ' + '\n  '.join(_ckpt_candidates)
print(f'Using checkpoint: {ckpt}')

# Clean v2 directories to prevent stale files
for d in ['results/samples_v2', 'results/evaluation_v2']:
    if os.path.exists(d):
        _shutil.rmtree(d)

def run_gen(extra_args, label):
    cmd = [
        'python', 'src/generate.py',
        '--config', 'configs/xl.yaml',
        '--checkpoint', ckpt,
        '--mup',
        '--max-tokens', '4096',
        '--device', 'cuda',
    ] + extra_args
    print(f'  {label}')
    subprocess.run(cmd)

# Unconditional top-p (matches v1 sampling: --top-p 0.95)
print('=== Unconditional top-p ===')
for temp in ['0.5', '0.8', '1.0']:
    run_gen([
        '--top-k', '0', '--top-p', '0.95',
        '--temperature', temp,
        '--num-samples', '10',
        '--output-dir', f'results/samples_v2/unconditional/topp_t{temp}',
    ], f'top-p temp={temp}')

# Unconditional top-k
print('\n=== Unconditional top-k ===')
for temp in ['0.5', '0.8', '1.0']:
    run_gen([
        '--top-k', '40',
        '--temperature', temp,
        '--num-samples', '10',
        '--output-dir', f'results/samples_v2/unconditional/topk_t{temp}',
    ], f'top-k temp={temp}')

# Prefix-conditioned
print('\n=== Prefix-conditioned ===')
for pname in ['face_partial', 'open_path', 'single_shape_group']:
    ptxt = open(f'results/prefixes_v2/{pname}.svg').read()
    run_gen([
        '--top-k', '0', '--top-p', '0.95',
        '--temperature', '0.8',
        '--num-samples', '5',
        '--prefix', ptxt,
        '--output-dir', f'results/samples_v2/prefix_conditioned/{pname}',
    ], f'prefix={pname}')

print('\nGeneration complete.')

In [ ]:
# Evaluation (v2 layout)
print('=== Evaluation ===')
subprocess.run([
    'python', 'src/evaluate.py',
    '--config', 'configs/xl.yaml',
    '--checkpoint', ckpt,
    '--mup',
    '--samples-dir', 'results/samples_v2',
    '--test-data', 'data/tokenized/test.bin',
    '--output-dir', 'results/evaluation_v2',
    '--device', 'cuda',
])

# Print results
if os.path.exists('results/evaluation_v2/eval_metrics.json'):
    with open('results/evaluation_v2/eval_metrics.json') as f:
        metrics = json.load(f)
    print(json.dumps(metrics, indent=2))

---
## 9. Summary

All results are saved under `results/` on Drive.

Next steps (on local machine):
1. `git pull` to get results
2. `python scripts/analyze_scaling.py` — SP power law fit
3. `python scripts/analyze_mup.py` — SP vs µP comparison plots
4. Write report

In [ ]:
print('\n' + '=' * 60)
print('PIPELINE COMPLETE')
print('=' * 60)

sections = [
    ('SP LR Sweep',    'results/runs/lr_sweep'),
    ('SP Scaling',     'results/runs/sp'),
    ('µP LR Sweep',   'results/runs/mup_lr_sweep'),
    ('µP Scaling',    'results/runs/mup'),
    ('Extended XL',    extended_dir),
    ('Samples v2',     'results/samples_v2'),
    ('Evaluation v2',  'results/evaluation_v2'),
]
for label, path in sections:
    exists = '✓' if os.path.exists(path) else '✗'
    print(f'  {exists} {label:>15s}: {path}')